# NDVI (HLS Landsat/Sentinel-2) per ISMN Station

<br>

For each station in `ismn_data/alaska/stations.geojson`, this notebook pulls NDVI from
NASA's **HLS** (Harmonized Landsat Sentinel-2) collections -- **HLSL30** (Landsat 8/9)
and **HLSS30** (Sentinel-2) -- at the Sentinel-1/NISAR acquisition dates already saved
by [`03_get_NISAR_GSLC_windows.ipynb`](03_get_NISAR_GSLC_windows.ipynb),
so the values can be plotted against SM-inversion residuals to check whether outliers
line up with vegetation cover. **This is a QC signal, not a modeling input** -- nothing
downstream of the inversion depends on it.

Unlike the NISAR GCOV/GSLC pipelines, no full-granule download or HDF5 remote-metadata
walk is needed: each HLS band is its own Cloud-Optimized GeoTIFF, so a windowed
`rasterio` read over HTTPS pulls only the pixels under an 80x80m station window (same
footprint used for the GSLC/Sentinel-1 windows) via a couple of HTTP range requests.

**Target dates, not a continuous time series.** For each station, NDVI is only pulled
for the SAR acquisition dates already on disk (via
[`util/nisar_coherence_closure.py`](../util/nisar_coherence_closure.py)'s
`load_station_acquisitions`, the same loader notebook 04 uses) -- the nearest available
HLS scene within `MAX_GAP_DAYS` of each target date is used, so this only needs to run
against the same dates the SM estimates already exist for.

Implemented by [`util/hls_ndvi.py`](../util/hls_ndvi.py).

:::{note} Requires notebook 03 to have already been run for this station set
Target dates come from `nisar_data/alaska_GSLC_windows/*.nc` -- if that directory is
empty or missing, every station will show 0 target dates.
:::

<hr>

## Overview

1. Load station locations
2. Get target (SAR acquisition) dates per station from the saved GSLC windows
3. Sanity-check HLS band/Fmask structure against real granules
4. Process each station: search HLSL30+HLSS30, match dates, compute window NDVI, save CSV
5. (Optional) Preview one station's output

<hr>

In [ ]:
import sys
import os
import json
from pathlib import Path
from datetime import datetime, timezone

import earthaccess

REPO_ROOT = os.path.dirname(os.getcwd())
if REPO_ROOT not in sys.path:
    sys.path.append(REPO_ROOT)

from util.nisar_station_timeseries import get_fsspec_https_session_with_timeout
from util.nisar_coherence_closure import load_station_acquisitions
from util.hls_ndvi import (
    inspect_hls_structure,
    search_hls_for_station,
    process_station_ndvi,
    DEFAULT_WINDOW_FOOTPRINT_M,
    DEFAULT_MAX_GAP_DAYS,
)

## 1. Load station locations

Same station list as the other notebooks in this cookbook.

In [ ]:
stations_path = Path.cwd() / "ismn_data" / "alaska" / "stations.geojson"

with open(stations_path) as f:
    stations_geojson = json.load(f)

stations = [
    {
        "name": feat["properties"]["station_val"],
        "lon": feat["geometry"]["coordinates"][0],
        "lat": feat["geometry"]["coordinates"][1],
    }
    for feat in stations_geojson["features"]
]

print(f"Loaded {len(stations)} stations")
stations[:3]

## 2. Target dates per station

Every SAR acquisition date already saved for a station, across all of its kept
tracks -- NDVI doesn't care which NISAR track an acquisition came from, only the
calendar date, so tracks aren't kept separate here the way they are for coherence
grouping in notebook 04.

In [ ]:
gslc_dir = Path.cwd() / "nisar_data" / "alaska_GSLC_windows"

target_dates_by_station = {}
for station in stations:
    groups = load_station_acquisitions(gslc_dir, station["name"])
    dates = sorted({acq["datetime"] for track_acqs in groups.values() for acq in track_acqs})
    target_dates_by_station[station["name"]] = dates
    print(f"{station['name']}: {len(dates)} target date(s)")

## 3. Sanity-check HLS band/Fmask structure against real granules

Opens one real HLSL30 and one real HLSS30 granule and prints their asset URLs plus
Red/NIR/Fmask shape/dtype/CRS, so the assumptions documented at the top of
`util/hls_ndvi.py` (band-name suffixes, Fmask bit layout) can be checked against real
data before running Section 4's full batch. The band-name suffixes (`B04`/`B05`/`B8A`/
`Fmask`) and short names (`HLSL30`/`HLSS30`) have already been confirmed against a live
CMR search; the Fmask **bit layout** (which bit means cloud vs. shadow vs. snow) has
not -- check the printed value distribution against what you'd expect for this scene
before trusting the cloud mask across every station.

In [ ]:
earthaccess.login()

# See util/hls_ndvi.py's module docstring for why this is safe to share across every
# call in this notebook (unlike the GSLC pipeline's fs, nothing here is ever abandoned
# mid-read, so there's no analog of that pipeline's concurrent-shared-session bug).
fs = get_fsspec_https_session_with_timeout()

TEMPORAL = ("2026-06-01", datetime.now(timezone.utc).strftime("%Y-%m-%d"))

_sanity_results = search_hls_for_station(stations[0], TEMPORAL)
print(f"Found {len(_sanity_results)} candidate HLS granule(s) for station '{stations[0]['name']}'")

_seen_collections = set()
for _r in _sanity_results:
    _native_id = _r["meta"]["native-id"]
    _collection = _native_id.split(".")[1]  # "L30" or "S30"
    if _collection in _seen_collections:
        continue
    _seen_collections.add(_collection)
    print(f"\n{'=' * 70}\n{_native_id}\n{'=' * 70}")
    inspect_hls_structure(_r, fs)

## 4. Process each station: search, match dates, compute NDVI, save CSV

For each station, in turn:

1. **Search** `HLSL30` + `HLSS30` using `point=(lon, lat)` over the full date range
   (`TEMPORAL`), combined and sorted by acquisition date.
2. For each of the station's **target dates** (Section 2), find the nearest available
   HLS scene; if the gap exceeds `MAX_GAP_DAYS`, the row is still recorded (for
   visibility) with `gap_days` showing how far off it actually is.
3. For the matched scene, **read the Red/NIR/Fmask bands** as an 80x80m window
   (`WINDOW_FOOTPRINT_M`) centered on the station's nearest pixel -- no full-scene
   download, just the windowed pixels via HTTP range requests.
4. **Mask** cloud/cloud-adjacent/cloud-shadow/snow-ice pixels (via Fmask) and exclude
   the reflectance fill value, then compute NDVI = (NIR-Red)/(NIR+Red) over whatever
   valid pixels remain in the window.
5. **Save** `{station_name}_HLS_ndvi.csv` under `output_dir`, with `n_valid_px`/
   `n_total_px` so a heavily-masked (mostly cloud) window is visible rather than
   silently averaged over just 1-2 pixels.

In [ ]:
WINDOW_FOOTPRINT_M = DEFAULT_WINDOW_FOOTPRINT_M  # 80m, matches the GSLC/Sentinel-1 window
MAX_GAP_DAYS = DEFAULT_MAX_GAP_DAYS

output_dir = Path.cwd() / "nisar_data" / "alaska_HLS_ndvi"

for station in stations:
    target_dates = target_dates_by_station[station["name"]]
    if not target_dates:
        print(f"{station['name']}: no target dates, skipping")
        continue

    search_results = search_hls_for_station(station, TEMPORAL)
    csv_path = process_station_ndvi(
        station, target_dates, search_results, output_dir, fs,
        window_footprint_m=WINDOW_FOOTPRINT_M, max_gap_days=MAX_GAP_DAYS,
    )
    print(f"{station['name']}: {len(search_results)} HLS scene(s) found, "
          f"{len(target_dates)} target date(s) -> {csv_path.name}")

## 5. (Optional) Preview one station's output

In [ ]:
import pandas as pd

preview_station = stations[0]["name"]
ndvi_df = pd.read_csv(output_dir / f"{preview_station}_HLS_ndvi.csv")
ndvi_df

<hr>

## Summary

You now have, under `nisar_data/alaska_HLS_ndvi/`, one CSV per station:
`{station_name}_HLS_ndvi.csv`, columns `target_date` (the SAR acquisition date),
`hls_date` (the matched HLS scene's date), `sensor` (`HLSL30`/`HLSS30`), `gap_days`,
`ndvi_mean`, `ndvi_std`, `n_valid_px`, `n_total_px`.

Join on `target_date` against whatever SM-inversion output uses the same acquisition
dates (e.g. `nisar_data/alaska_GSLC_coherence_closure/{station}_NISAR_GSLC_closure_phase.csv`'s
`date_1`/`date_2`/`date_3`) to plot NDVI alongside inversion residuals. A row with
`n_valid_px` close to 0 (heavily cloud/shadow/snow-masked) or a large `gap_days` should
be treated as unreliable rather than a genuine vegetation signal.

Re-running this notebook after more GSLC windows are downloaded (more target dates) or
more time has passed (more HLS scenes available) is safe -- both CSVs are fully rebuilt
from scratch each run, purely from already-searched data plus fresh windowed reads.

<hr>